# 02 — Preprocessing Pipeline
**Brugada-HUCA ECG Dataset**

Goals:
1. Apply the full filter chain (bandpass → notch) to all 363 records
2. Z-score normalize each lead independently
3. Create stratified **train / val / test** splits (70 / 15 / 15 %)
4. Analyse and handle the class imbalance (pos_weight + optional SMOTE)
5. Save processed arrays to `data/processed/` as compressed `.npz` files
6. Verify reproducibility

## 0 — Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'src' / 'config.py').exists()),
    _cwd,
)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import config as cfg
from preprocessing import (
    load_all_records,
    filter_signal, bandpass_filter, notch_filter,
    normalize_signal, preprocess_batch,
    make_splits, compute_pos_weight,
    save_processed, load_processed,
    augment_signal,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

print('ROOT:', ROOT)
print('Processed output:', cfg.DATA_PROCESSED)

## 1 — Load Raw Signals

In [ ]:
meta = pd.read_csv(cfg.METADATA_FILE)

print('Loading all records...')
X_raw, y, ids = load_all_records(meta, cfg.DATA_RAW, verbose=True)

print(f'\nX_raw : {X_raw.shape}  dtype={X_raw.dtype}')
print(f'y     : {y.shape}  Brugada={( y==1).sum()}  Normal={(y==0).sum()}')

## 2 — Filtering: Visualise the Effect

Show one Brugada signal before and after each filter stage.

In [ ]:
# Pick a Brugada sample for illustration
b_idx = np.where(y == 1)[0][0]
raw   = X_raw[b_idx]         # (12, 1200)

bp    = bandpass_filter(raw)  # after bandpass
ntch  = notch_filter(bp)      # after notch
norm  = normalize_signal(ntch) # after normalization

lead_name = 'V1'
li = cfg.LEAD_NAMES.index(lead_name)
time = np.arange(cfg.N_SAMPLES) / cfg.FS

stages = [
    (raw[li],  'Raw',              '#888888'),
    (bp[li],   'After Bandpass',   '#1565C0'),
    (ntch[li], 'After Notch',      '#2E7D32'),
    (norm[li], 'After Normalize',  '#C62828'),
]

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
fig.suptitle(f'Preprocessing Stages — Lead {lead_name} (Brugada ID: {ids[b_idx]})', fontsize=13, fontweight='bold')

for ax, (signal, label, color) in zip(axes, stages):
    ax.plot(time, signal, color=color, linewidth=0.9)
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, alpha=0.4)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(REPORTS / 'preproc_filter_stages.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Power Spectral Density — before vs after filtering
from scipy.signal import welch

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sig, title, color in [
    (axes[0], raw[li],  f'Raw — {lead_name}',       '#888888'),
    (axes[1], ntch[li], f'Filtered — {lead_name}',  '#2E7D32'),
]:
    freqs, psd = welch(sig, fs=cfg.FS, nperseg=256)
    ax.semilogy(freqs, psd, color=color, linewidth=1.2)
    ax.axvspan(0, cfg.LOWCUT,    alpha=0.12, color='red',   label='Removed (bandpass)')
    ax.axvspan(cfg.HIGHCUT, 50, alpha=0.12, color='red')
    ax.axvline(cfg.NOTCH_FREQ,   color='orange', linestyle='--', linewidth=1, label='Notch (50 Hz)')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD (mV²/Hz)')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Power Spectral Density — Before vs After Filtering', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'preproc_psd_comparison.png', dpi=150)
plt.show()

## 3 — Apply Full Pipeline to All Records

In [ ]:
from tqdm.notebook import tqdm

print('Preprocessing all records...')
X_clean = preprocess_batch(X_raw)

print(f'\nX_clean : {X_clean.shape}  dtype={X_clean.dtype}')
print(f'  mean  = {X_clean.mean():.6f}  (expect ~0)')
print(f'  std   = {X_clean.std():.6f}   (expect ~1)')

In [ ]:
# Verify normalization: per-lead statistics
lead_means = X_clean.mean(axis=(0, 2))  # (12,)
lead_stds  = X_clean.std(axis=(0, 2))   # (12,)

df_stats = pd.DataFrame({'lead': cfg.LEAD_NAMES, 'mean': lead_means, 'std': lead_stds})
df_stats = df_stats.set_index('lead').round(4)
print('Per-lead statistics after normalization:')
display(df_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(cfg.LEAD_NAMES, lead_means, color='steelblue', edgecolor='black')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Per-Lead Mean (should be ~0)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(cfg.LEAD_NAMES, lead_stds, color='darkorange', edgecolor='black')
axes[1].axhline(1, color='red', linestyle='--')
axes[1].set_title('Per-Lead Std (should be ~1)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(REPORTS / 'preproc_normalization_check.png', dpi=150)
plt.show()

## 4 — Train / Val / Test Split

In [ ]:
splits = make_splits(X_clean, y, ids)

print('Split sizes:')
for name in ('train', 'val', 'test'):
    X_s = splits[f'X_{name}']
    y_s = splits[f'y_{name}']
    n_b = (y_s == 1).sum()
    n_n = (y_s == 0).sum()
    print(f'  {name:5s}: {len(y_s):3d} samples  |  Brugada={n_b}  Normal={n_n}  '
          f'({100*n_b/len(y_s):.1f}% positive)')

In [ ]:
# Visualise split class balance
split_names = ['train', 'val', 'test']
brugada_counts = [(splits[f'y_{s}'] == 1).sum() for s in split_names]
normal_counts  = [(splits[f'y_{s}'] == 0).sum() for s in split_names]

x = np.arange(3)
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars_b = ax.bar(x - width/2, brugada_counts, width, label='Brugada', color='#C62828', edgecolor='black')
bars_n = ax.bar(x + width/2, normal_counts,  width, label='Normal',  color='#1565C0', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in split_names])
ax.set_ylabel('Count')
ax.set_title('Class Distribution per Split (stratified)')
ax.legend()

for bars in (bars_b, bars_n):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS / 'preproc_split_distribution.png', dpi=150)
plt.show()

## 5 — Class Imbalance: Strategy

Two complementary approaches:
- **`pos_weight`** in `BCEWithLogitsLoss` — always applied during training
- **SMOTE** — optional oversampling on the training set only

In [ ]:
pos_w = compute_pos_weight(splits['y_train'])
print(f'pos_weight for BCEWithLogitsLoss: {pos_w:.3f}')
print(f'  → Each Brugada sample contributes {pos_w:.1f}x more to the loss')

In [ ]:
# Optional: SMOTE oversampling
# Uncomment the block below if imbalanced-learn is installed.
# Note: SMOTE is applied ONLY to training data — never val or test.

USE_SMOTE = False  # flip to True to enable

if USE_SMOTE:
    from preprocessing import smote_oversample
    X_train_sm, y_train_sm = smote_oversample(splits['X_train'], splits['y_train'])
    splits['X_train'] = X_train_sm
    splits['y_train'] = y_train_sm
    print(f'After SMOTE — train size: {len(y_train_sm)}  Brugada: {(y_train_sm==1).sum()}  Normal: {(y_train_sm==0).sum()}')
else:
    print('SMOTE disabled. Will use pos_weight in loss function.')
    print(f'Training set: {len(splits["y_train"])} samples  Brugada={( splits["y_train"]==1).sum()}  Normal={(splits["y_train"]==0).sum()}')

## 6 — Data Augmentation Preview

In [ ]:
rng = np.random.default_rng(cfg.RANDOM_SEED)
original = splits['X_train'][0]                        # one training signal
augmented = augment_signal(original, rng=rng)

v1_idx = cfg.LEAD_NAMES.index('V1')
time = np.arange(cfg.N_SAMPLES) / cfg.FS

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(time, original[v1_idx],  color='steelblue', linewidth=0.9, label='Original')
axes[0].set_title('Original (Lead V1)')
axes[0].set_ylabel('Amplitude (normalized)')

axes[1].plot(time, augmented[v1_idx], color='darkorange', linewidth=0.9, label='Augmented')
axes[1].set_title('Augmented: noise + shift + scale (Lead V1)')
axes[1].set_ylabel('Amplitude (normalized)')
axes[1].set_xlabel('Time (s)')

plt.suptitle('Augmentation Preview', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'preproc_augmentation_preview.png', dpi=150)
plt.show()

## 7 — Save Processed Data

In [ ]:
print('Saving processed splits...')
save_processed(splits, cfg.DATA_PROCESSED)
print('Done.')

## 8 — Reproducibility Verification

Reload each split from disk and confirm shapes and label distributions match.

In [ ]:
print('Reloading from disk to verify...')
for split_name in ('train', 'val', 'test'):
    X_r, y_r, ids_r = load_processed(split_name, cfg.DATA_PROCESSED)
    X_o = splits[f'X_{split_name}']
    y_o = splits[f'y_{split_name}']

    shape_ok = (X_r.shape == X_o.shape) and (y_r.shape == y_o.shape)
    values_ok = np.allclose(X_r, X_o) and np.array_equal(y_r, y_o)
    print(f'  {split_name:5s}: shape={X_r.shape}  shape_match={shape_ok}  values_match={values_ok}')

## 9 — Summary

| Step | Setting |
|---|---|
| Bandpass filter | 0.5–40 Hz, Butterworth order 4, zero-phase |
| Notch filter | 50 Hz (EU powerline), Q=30 |
| Normalization | Z-score per lead per patient |
| Split | 70 / 15 / 15 %, stratified, seed=42 |
| Imbalance | pos_weight = n_neg / n_pos applied in loss |
| Augmentation | Gaussian noise (σ=0.01), time-shift (±10), scale (0.9–1.1) |
| Output | `data/processed/{train,val,test}.npz` |

**Next step:** `03_training.ipynb` — define the 1D ResNet and run training.